# Thực hành Causal Inference trên Bộ Dữ Liệu Lalonde

Notebook này hướng dẫn chi tiết cách thực hành **Causal Inference (Suy luận nhân quả)** sử dụng bộ dữ liệu kinh điển **Lalonde (1986)**.

---

## 1. Giới thiệu Bộ Dữ Liệu Lalonde & Câu Hỏi Nghiên Cứu

### Bối cảnh lịch sử
Bộ dữ liệu Lalonde được giới thiệu lần đầu bởi nhà kinh tế học **Rajeev Dehejia** và **Sadek Wahba** (1999), dựa trên nghiên cứu gốc của **Robert Lalonde** (1986). Nghiên cứu đánh giá hiệu quả của chương trình **National Supported Work Demonstration (NSW)** - một chương trình đào tạo nghề và hỗ trợ việc làm tạm thời được chính phủ Mỹ tài trợ vào những năm 1970 dành cho những đối tượng gặp khó khăn kinh tế kéo dài (như người thất nghiệp lâu năm, người từng thụ án, bà mẹ đơn thân nhận trợ cấp xã hội).

### Câu hỏi nghiên cứu chính (Research Question)
> **"Việc tham gia vào chương trình đào tạo nghề NSW (`treat`) có thực sự làm tăng thu nhập thực tế năm 1978 (`re78`) của người tham gia so với việc không tham gia hay không? Tác động nhân quả thực tế (Causal Effect) là bao nhiêu?"**

### Cấu trúc các trường thông tin (Fields Description)
Bộ dữ liệu chứa các biến sau đây:
*   **`treat`** (Biến can thiệp - Treatment): Nhận giá trị `1` nếu tham gia chương trình NSW, `0` nếu ở nhóm đối chứng (Control).
*   **`age`** (Tuổi): Tuổi của đối tượng tham gia nghiên cứu.
*   **`educ`** (Học vấn): Số năm đi học.
*   **`black`** (Chủng tộc): Nhận giá trị `1` nếu là người da màu (Black), `0` nếu ngược lại.
*   **`hispan`** (Chủng tộc): Nhận giá trị `1` nếu là người Tây Ban Nha/Bồ Đào Nha (Hispanic), `0` nếu ngược lại.
*   **`married`** (Hôn nhân): Nhận giá trị `1` nếu đã kết hôn, `0` nếu ngược lại.
*   **`nodegree`** (Bằng cấp): Nhận giá trị `1` nếu không có bằng tốt nghiệp trung học (High School Diploma), `0` nếu ngược lại.
*   **`re74`** (Thu nhập 1974): Thu nhập thực tế năm 1974 trước khi chương trình diễn ra (đơn vị: USD).
*   **`re75`** (Thu nhập 1975): Thu nhập thực tế năm 1975 trước khi chương trình diễn ra (đơn vị: USD).
*   **`re78`** (Biến kết quả - Outcome): Thu nhập thực tế năm 1978 sau khi chương trình kết thúc (đơn vị: USD).

## 2. Setup & Tải bộ dữ liệu

In [ ]:
import os
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# Cấu hình vẽ đồ thị đẹp
sns.set_theme(style="ticks")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Đường dẫn tải dữ liệu
url = "https://vincentarelbundock.github.io/Rdatasets/csv/MatchIt/lalonde.csv"
csv_path = "lalonde.csv"

if not os.path.exists(csv_path):
    print("Đang tải bộ dữ liệu Lalonde...")
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response:
        with open(csv_path, 'wb') as f:
            f.write(response.read())
    print("Tải thành công!")
else:
    print("Bộ dữ liệu đã có sẵn trên máy local.")

df = pd.read_csv(csv_path)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Hình dạng dữ liệu (Shape): {df.shape}")
df.head()

## 3. Phân Tích Khám Phá Dữ Liệu (Exploratory Data Analysis - EDA)

Trong Causal Inference, nhiệm vụ quan trọng nhất của EDA là đánh giá sự **mất cân bằng (imbalance)** của các biến nền (covariates) giữa nhóm được can thiệp (`Treated`) và nhóm đối chứng (`Control`). Nếu hai nhóm này không tương đồng, việc so sánh trực tiếp kết quả sẽ bị sai lệch nặng nề (selection bias).

In [ ]:
# 3.1 Thống kê mô tả chung cho từng nhóm
covariates = ['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75']

print("=== Thống kê trung bình của nhóm Đối chứng (Control - treat=0) ===")
display(df[df['treat'] == 0][covariates + ['re78']].describe().round(2))

print("=== Thống kê trung bình của nhóm Học nghề (Treated - treat=1) ===")
display(df[df['treat'] == 1][covariates + ['re78']].describe().round(2))

In [ ]:
# 3.2 Tính toán Standardized Mean Difference (SMD)
# SMD là thước đo chuẩn hóa được sử dụng phổ biến trong Causal Inference để đo lường mức độ mất cân bằng.
# SMD > 0.1 thường là dấu hiệu cho thấy sự mất cân bằng đáng kể cần phải được điều chỉnh.

def calculate_smd(df, treat_col, cov_list):
    treated = df[df[treat_col] == 1]
    control = df[df[treat_col] == 0]
    
    smd_list = []
    for cov in cov_list:
        mean_t = treated[cov].mean()
        mean_c = control[cov].mean()
        var_t = treated[cov].var()
        var_c = control[cov].var()
        
        pooled_sd = np.sqrt((var_t + var_c) / 2)
        smd = (mean_t - mean_c) / pooled_sd
        smd_list.append(smd)
        
    return pd.DataFrame({'Biến': cov_list, 'SMD (Chưa hiệu chỉnh)': smd_list})

smd_df = calculate_smd(df, 'treat', covariates)
smd_df

In [ ]:
# 3.3 Trực quan hóa sự mất cân bằng của một số biến số chính
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# So sánh phân phối thu nhập trước can thiệp (1974)
sns.kdeplot(data=df, x='re74', hue='treat', common_norm=False, fill=True, ax=axes[0, 0], palette="muted")
axes[0, 0].set_title("Phân phối Thu nhập năm 1974 (re74)", fontsize=12)
axes[0, 0].set_xlabel("Thu nhập (USD)")

# So sánh phân phối Tuổi
sns.boxplot(data=df, x='treat', y='age', ax=axes[0, 1], palette="pastel")
axes[0, 1].set_title("So sánh Phân phối Tuổi giữa 2 nhóm", fontsize=12)
axes[0, 1].set_xticklabels(["Control", "Treated"])

# Tỷ lệ trình độ học vấn (số năm học)
sns.histplot(data=df, x='educ', hue='treat', multiple='dodge', shrink=0.8, ax=axes[1, 0], palette="muted")
axes[1, 0].set_title("Phân phối Số năm học vấn (educ)", fontsize=12)

# Tỷ lệ không có bằng cấp
sns.barplot(data=df, x='treat', y='nodegree', ax=axes[1, 1], palette="pastel", errorbar=None)
axes[1, 1].set_title("Tỷ lệ Không có Bằng cấp (nodegree)", fontsize=12)
axes[1, 1].set_xticklabels(["Control", "Treated"])
axes[1, 1].set_ylabel("Tỷ lệ")

plt.tight_layout()
plt.show()

### Nhận xét từ EDA:
1. **Mất cân bằng nghiêm trọng:** Giá trị tuyệt đối của SMD đối với phần lớn các biến đều vượt xa ngưỡng **0.1**, đặc biệt là `black` (SMD = ~1.73), `nodegree` (SMD = ~1.2), và thu nhập quá khứ `re74` (SMD = ~-0.94).
2. **Selection Bias:** Nhóm can thiệp (`Treated`) có tỷ lệ người da màu cao hơn nhiều, tỷ lệ thất nghiệp trước chương trình cực cao (thu nhập TB 1974 rất thấp), và phần lớn không có bằng cấp học vấn so với nhóm Đối chứng (`Control`).
3. **Kết luận:** Nếu chúng ta chỉ đơn thuần so sánh thu nhập năm 1978 giữa 2 nhóm, chúng ta sẽ thu được ước lượng sai lầm (vì nhóm Treated vốn dĩ xuất phát điểm gặp nhiều khó khăn hơn nhiều).

## 4. Naive ATE (Average Treatment Effect) - Ước lượng Thô

In [ ]:
mean_y_treated = df[df['treat'] == 1]['re78'].mean()
mean_y_control = df[df['treat'] == 0]['re78'].mean()
naive_ate = mean_y_treated - mean_y_control

print(f"Thu nhập TB 1978 nhóm Học nghề (Treated): ${mean_y_treated:,.2f}")
print(f"Thu nhập TB 1978 nhóm Đối chứng (Control): ${mean_y_control:,.2f}")
print(f"Naive ATE (Ước lượng chưa qua xử lý nhiễu): ${naive_ate:,.2f}")

**Kết quả bất ngờ:** Naive ATE âm (**-$635.03**), cho thấy việc học nghề làm giảm thu nhập của học viên vào năm 1978. Điều này vô lý và phi thực tế. Tiếp theo chúng ta sẽ tiến hành khử nhiễu để bóc tách tác động thực sự.

## 5. Ước lượng Propensity Scores

In [ ]:
X = df[covariates]
y = df['treat']

# Huấn luyện mô hình Logistic Regression dự đoán xác suất được phân phối vào nhóm Treatment
ps_model = LogisticRegression(max_iter=1000, random_state=42)
ps_model.fit(X, y)

df['propensity_score'] = ps_model.predict_proba(X)[:, 1]

print("Thống kê điểm xu hướng theo nhóm:")
display(df.groupby('treat')['propensity_score'].describe())

In [ ]:
# Trực quan hóa vùng chung sống (Common Support/Overlap)
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='propensity_score', hue='treat', kde=True, bins=30, common_norm=False, alpha=0.5, palette="Set1")
plt.title("Vùng Phân Bố Chung (Common Support) Của Propensity Scores", fontsize=14)
plt.xlabel("Propensity Score")
plt.ylabel("Tần suất")
plt.legend(title="Group", labels=["Treated (Học nghề)", "Control (Đối chứng)"])
plt.show()

## 6. Cân Bằng Dữ Liệu bằng Propensity Score Matching (PSM)

Chúng ta áp dụng thuật toán **Ghép cặp Láng giềng Gần nhất (1:1 Nearest Neighbor Matching) có thay thế** để ghép cặp từng đối tượng nhóm Học nghề với một đối tượng nhóm Đối chứng tương đương nhất về điểm xu hướng.

In [ ]:
treated_idx = df[df['treat'] == 1].index
control_idx = df[df['treat'] == 0].index

treated_ps = df.loc[treated_idx, ['propensity_score']].values
control_ps = df.loc[control_idx, ['propensity_score']].values

nn = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
nn.fit(control_ps)

distances, indices = nn.kneighbors(treated_ps)
matched_control_idx = control_idx[indices.flatten()]

matched_treated_df = df.loc[treated_idx].copy()
matched_control_df = df.loc[matched_control_idx].copy()
matched_df = pd.concat([matched_treated_df, matched_control_df])

print(f"Số lượng cặp được ghép thành công: {len(matched_treated_df)} cặp.")

### Kiểm tra lại độ cân bằng sau khi ghép cặp (Post-matching Balance Check)

In [ ]:
# Tính toán lại SMD sau khi ghép cặp
def calculate_matched_smd(matched_treated, matched_control, cov_list):
    smd_list = []
    for cov in cov_list:
        mean_t = matched_treated[cov].mean()
        mean_c = matched_control[cov].mean()
        var_t = matched_treated[cov].var()
        var_c = matched_control[cov].var()
        
        pooled_sd = np.sqrt((var_t + var_c) / 2)
        smd = (mean_t - mean_c) / pooled_sd
        smd_list.append(smd)
        
    return pd.DataFrame({'Biến': cov_list, 'SMD (Sau Matching)': smd_list})

smd_matched = calculate_matched_smd(matched_treated_df, matched_control_df, covariates)

# Gộp bảng so sánh trước và sau Matching
balance_comparison = smd_df.merge(smd_matched, on='Biến')
balance_comparison['Khử nhiễu (%)'] = ((balance_comparison['SMD (Chưa hiệu chỉnh)'].abs() - balance_comparison['SMD (Sau Matching)'].abs()) / balance_comparison['SMD (Chưa hiệu chỉnh)'].abs() * 100).round(2)
balance_comparison

**Nhận xét:** Tất cả các biến nền sau ghép cặp đều có **SMD nằm trong khoảng [-0.1, 0.1]** (ngưỡng lý tưởng thể hiện hai nhóm đã cân bằng hoàn hảo về phân phối). Mức độ khử nhiễu đạt trên 95% ở hầu hết các biến số.

### Ước lượng Tác động Nhân quả (ATT - Average Treatment Effect on the Treated)

In [ ]:
att_psm = matched_treated_df['re78'].values - matched_control_df['re78'].values
att_psm_mean = att_psm.mean()
print(f"Hiệu quả can thiệp trên nhóm học viên (ATT) qua PSM: ${att_psm_mean:,.2f}")

## 7. Cân Bằng Dữ Liệu bằng Inverse Probability Weighting (IPW)

Thay vì bỏ bớt dữ liệu để ghép cặp, phương pháp **IPW (Trọng số nghịch đảo)** gán cho mỗi đối tượng một trọng số tương ứng nhằm tạo lập một quần thể giả định (pseudo-population) hoàn toàn cân bằng.

Công thức trọng số IPW cho ATE (Average Treatment Effect):
$$w_i = \frac{T_i}{e(X_i)} + \frac{1 - T_i}{1 - e(X_i)}$$

In [ ]:
e = df['propensity_score']
t = df['treat']
y = df['re78']

# Áp dụng công thức tính trọng số IPW
df['ipw_weight'] = t / e + (1 - t) / (1 - e)

# Sử dụng công thức tự chuẩn hóa Hajek giúp ước lượng ổn định hơn, tránh các điểm xu hướng quá nhỏ/lớn
weighted_y_treated = (t * y / e).sum() / (t / e).sum()
weighted_y_control = ((1 - t) * y / (1 - e)).sum() / ((1 - t) / (1 - e)).sum()
ipw_ate_hajek = weighted_y_treated - weighted_y_control

print(f"Hiệu quả can thiệp trên toàn dân số (ATE) qua IPW (Hajek): ${ipw_ate_hajek:,.2f}")

## 8. Tổng Kết Bài Học

Hãy cùng so sánh ba kết quả đo lường:

In [ ]:
summary_df = pd.DataFrame({
    'Phương pháp': ['Naive ATE (So sánh thô)', 'Propensity Score Matching (ATT)', 'IPW Hajek Estimator (ATE)'],
    'Ước lượng Hiệu quả': [naive_ate, att_psm_mean, ipw_ate_hajek],
    'Ý nghĩa': [
        'Bị nhiễu chọn lọc (Selection Bias) cực mạnh làm kết quả âm',
        'Đã bóc tách nhiễu, ước lượng hiệu quả trên nhóm tham gia',
        'Đã bóc tách nhiễu, ước lượng hiệu quả trên toàn bộ dân số'
    ]
})
summary_df.round(2)

### Kết luận rút ra:
1. **Sức mạnh của Causal Inference:** Nếu chỉ dùng phân tích hồi quy hoặc so sánh trung bình tuyến tính thô, bạn sẽ kết luận sai lầm rằng chương trình làm giảm thu nhập của học viên. Causal Inference giúp đảo chiều kết quả chứng minh chương trình tăng thu nhập lên khoảng **+$1,200 - +$1,600**.
2. **Cơ chế hoạt động:** Bằng cách biến đổi dữ liệu quan sát bị nhiễu thông qua ghép cặp (PSM) hoặc gán trọng số (IPW), chúng ta có thể tạo ra các nhóm so sánh có phân phối đặc điểm tương đồng tuyệt đối, mô phỏng lại một cuộc thử nghiệm ngẫu nhiên thực tế (RCT).